In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data_processed_credit_data_cleaned_v2.csv")


In [2]:
df["debt_to_income"] = df["loan_amount"] / df["annual_income"]
df["debt_to_income"] = df["debt_to_income"].clip(upper=5)


In [3]:
df["utilization_stress"] = df["credit_utilization"] ** 2

In [4]:
df["exposure_ratio"] = df["loan_amount"] / df["credit_limit"]
df["exposure_ratio"] = df["exposure_ratio"].clip(upper=1.5)


In [5]:
delay_cols = [f"pay_delay_m{i}" for i in range(1, 7)]
df["avg_payment_delay"] = df[delay_cols].mean(axis=1)


In [6]:
df["max_payment_delay"] = df[delay_cols].max(axis=1)


In [7]:
df["recent_delinquency_flag"] = (
    (df["pay_delay_m1"] > 0) | (df["pay_delay_m2"] > 0)
).astype(int)


In [8]:
pay_cols = [f"pay_amt_m{i}" for i in range(1, 4)]
bill_cols = [f"bill_amt_m{i}" for i in range(1, 4)]

df["payment_to_bill_ratio"] = (
    df[pay_cols].sum(axis=1) / df[bill_cols].sum(axis=1)
).replace([np.inf, -np.inf], 0)

df["payment_to_bill_ratio"] = df["payment_to_bill_ratio"].clip(upper=2)


In [9]:
df["missed_payment_count"] = (df[pay_cols + bill_cols] == 0).sum(axis=1)


In [10]:
df["employment_stability"] = np.log1p(df["employment_years"])


In [11]:
df["has_past_default"] = (df["past_defaults"] > 0).astype(int)


In [12]:
df["stress_interaction"] = (
    df["utilization_stress"] *
    df["recent_delinquency_flag"]
)


In [13]:
engineered_features = [
    "debt_to_income", "utilization_stress", "exposure_ratio",
    "avg_payment_delay", "max_payment_delay",
    "recent_delinquency_flag", "payment_to_bill_ratio",
    "missed_payment_count", "employment_stability",
    "has_past_default", "stress_interaction"
]

df[engineered_features].describe().T


,count,mean,std,min,25%,50%,75%,max
debt_to_income,100000.0,0.513416,0.302106,0.001362,0.282802,0.456850,0.698921,2.215740
utilization_stress,100000.0,0.438865,0.345634,0.010001,0.131761,0.360550,0.697290,1.209963
exposure_ratio,100000.0,0.715971,0.266835,0.154314,0.481615,0.763941,1.000000,1.000000
avg_payment_delay,100000.0,0.627600,0.496983,0.000000,0.166667,0.500000,1.000000,3.666667
max_payment_delay,100000.0,2.381780,1.705442,0.000000,1.000000,2.000000,3.000000,6.000000
recent_delinquency_flag,100000.0,0.513910,0.499809,0.000000,0.000000,1.000000,1.000000,1.000000
payment_to_bill_ratio,99737.0,0.746956,0.259493,0.000000,0.570771,0.719497,0.894614,2.000000
missed_payment_count,100000.0,0.495030,0.994296,0.000000,0.000000,0.000000,0.000000,6.000000
employment_stability,100000.0,2.718116,0.915185,0.000000,2.397895,2.995732,3.401197,3.688879
has_past_default,100000.0,0.186150,0.389230,0.000000,0.000000,0.000000,0.000000,1.000000


In [14]:
df.to_csv("data_processed_credit_data_features_v2.csv", index=False)
